# Signal Extraction — Comparative Analysis

**MSc Course Assignment** | Author: MSC Student | Date: 2026-05-07

---

## 1. Problem Statement

A composite signal $\sigma(t)$ is formed by summing four sinusoidal components:

$$\sigma[n] = \sum_{i=1}^{4} S_i[n], \qquad S_i[n] = A_i \sin\!\left(2\pi f_i \frac{n}{f_s} + \phi_i\right)$$

where $f_s = 1000\,\text{Hz}$, $N = f_s \times T = 10{,}000$ samples, and
$\{f_i\} = \{10,\,50,\,120,\,300\}\,\text{Hz}$.

Each sinusoid is independently corrupted with amplitude noise (coefficient $\alpha$) and phase noise (coefficient $\beta$), both drawn from $\mathcal{N}(0,1)$:

$$\tilde{S}_i[n] = \bigl(A_i + \alpha\,\varepsilon_A^{(i)}\bigr)\,\sin\!\left(2\pi f_i \frac{n}{f_s} + \phi_i + \beta\,\varepsilon_\phi^{(i)}\right)$$

The noisy composite sum is:

$$\tilde{\sigma}[n] = \sum_{i=1}^{4} \tilde{S}_i[n]$$

### Dataset sample structure

For each training sample a random window start $t$ and target index $c$ are chosen:

| Part | Shape | Content |
|------|-------|---------|
| Selector $\mathbf{c}$ | $(4,)$ | One-hot vector identifying target frequency |
| Noisy window | $(10,)$ | $\tilde{\sigma}[t:t+10]$ |
| **Input** $\mathbf{x}$ | $(14,)$ | $[\mathbf{c}\;\|\;\tilde{\sigma}[t:t+10]]$ |
| **Target** $\mathbf{y}$ | $(10,)$ | $S_c[t:t+10]$ |

Training minimises the Mean Squared Error:

$$\mathcal{L}(\theta) = \frac{1}{W}\sum_{k=0}^{W-1}\bigl(\hat{y}_k - y_k\bigr)^2, \qquad W=10$$

Three architectures are compared: **Fully Connected (FC)**, **Vanilla RNN**, and **LSTM**.

---

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Paths — notebook lives in notebooks/, project root is one level up
ROOT    = os.path.dirname(os.getcwd())
SRC     = os.path.join(ROOT, 'src')
RESULTS = os.path.join(ROOT, 'results')
ASSETS  = os.path.join(ROOT, 'assets')
CONFIG  = os.path.join(ROOT, 'config', 'setup.json')

sys.path.insert(0, SRC)

MODELS      = ['fc', 'rnn', 'lstm']
FREQ_LABELS = ['10 Hz', '50 Hz', '120 Hz', '300 Hz']

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

def load_json(path):
    with open(path) as f:
        return json.load(f)

print('Root:', ROOT)
print('Results dir exists:', os.path.isdir(RESULTS))
print('Assets dir exists: ', os.path.isdir(ASSETS))

---
## 2. Signal Overview

The four sinusoids span two decades of frequency (10–300 Hz).
At $f_s = 1000\,\text{Hz}$ the Nyquist limit is $f_{\max} = 500\,\text{Hz}$; all
components satisfy the Nyquist–Shannon criterion $f_i < f_s/2$.

The 10-sample context window ($10\,\text{ms}$ at 1 kHz) covers:

| Frequency | Cycles per window | Character |
|-----------|-------------------|-----------|
| 10 Hz  | 0.10 | Slow trend — less than one full period visible |
| 50 Hz  | 0.50 | Half-cycle — rising or falling slope |
| 120 Hz | 1.20 | One full cycle + slight overhang |
| 300 Hz | 3.00 | Three full cycles — highly distinctive pattern |

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(ASSETS, 'signal_overview.png')))

---
## 3. Baseline Comparison (EXP-01)

**Protocol:** FC, RNN, and LSTM trained with identical hyperparameters over
3 random seeds $\{42, 123, 777\}$.
Default config: $\alpha=0.1$, $\beta=0.1$, `hidden_size=128`, `n_layers=2`,
`lr=0.001`, `patience=10`, `epochs=100`.
Results are mean ± std across seeds.

In [ ]:
baseline = load_json(os.path.join(RESULTS, 'baseline', 'summary.json'))['models']

header = f"{'Model':<6}  {'Overall MSE (mean±std)':>24}  " + \
         '  '.join(f'{fl:>12}' for fl in FREQ_LABELS)
print(header)
print('-' * len(header))
for m in MODELS:
    mo = baseline[m]['mse_overall']
    pf = baseline[m]['mse_per_freq']
    freq_vals = '  '.join(f"{pf[str(k)]['mean']:.2e}" for k in range(4))
    print(f"{m.upper():<6}  {mo['mean']:.2e} ± {mo['std']:.2e}          {freq_vals}")

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'training_curves.png')))

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'mse_comparison.png')))

### 3.1 Discussion

**FC achieves the lowest overall MSE** under default hyperparameters.
This is counter-intuitive but explained by the input representation:

1. **The full window is already a flat feature vector.** The 14-dimensional input
   $[\mathbf{c}\,\|\,\tilde{\sigma}_{t:t+W}]$ encodes all available context.
   FC can learn the extraction mapping with a direct linear path; RNN/LSTM
   add recurrent overhead (gate matrices, hidden-state initialisation) that
   provides no benefit at `seq_len = 1`.

2. **Sinusoids are stationary.** A clean sinusoidal window is fully determined
   by its frequency, amplitude, and phase at $t=0$. There is no
   long-range temporal dependence to exploit.

3. **Vanishing gradients hurt RNN most.** The vanilla RNN update rule:

   $$\mathbf{h}_t = \tanh(W_{ih}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b})$$

   relies on $W_{hh}$ to propagate gradients. When $\|W_{hh}\| < 1$ gradients
   decay exponentially with depth/sequence length. LSTM mitigates this with
   its additive cell-state update:

   $$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t$$

   explaining why LSTM outperforms RNN in every experiment.

**Training dynamics:** All models converge within ≈ 40 epochs; early stopping
($\text{patience}=10$) triggers before the 100-epoch budget in every run.

---
## 4. Predicted vs. Clean Signal Examples

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'signal_examples.png')))

All three models reconstruct the clean sinusoidal window closely.
The aggregate MSE differences manifest as small amplitude/phase offsets in
individual windows. FC predictions are visually indistinguishable from the
ground truth.

---
## 5. Per-Frequency MSE Analysis (EXP-07)

**Research question (RQ-5):** Does RNN degrade more sharply at high frequencies
than LSTM, consistent with the vanishing-gradient hypothesis?

Define the **MSE frequency ratio**:

$$R_f = \frac{\text{MSE}(300\,\text{Hz})}{\text{MSE}(10\,\text{Hz})}$$

A ratio $R_f \gg 1$ for RNN (but $\approx 1$ for LSTM) would confirm the hypothesis.

In [ ]:
print(f"{'Model':<6}  {'MSE(10 Hz)':>12}  {'MSE(300 Hz)':>12}  {'Ratio':>8}  Interpretation")
print('-' * 65)
for m in MODELS:
    pf = baseline[m]['mse_per_freq']
    low   = pf['0']['mean']   # 10 Hz
    high  = pf['3']['mean']   # 300 Hz
    ratio = high / low if low > 0 else float('inf')
    interp = ('harder at 300 Hz' if ratio > 1.2
               else ('easier at 300 Hz' if ratio < 0.8 else 'similar'))
    print(f"{m.upper():<6}  {low:>12.2e}  {high:>12.2e}  {ratio:>8.2f}  {interp}")

In [ ]:
x = np.arange(len(FREQ_LABELS))
width = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (m, col) in enumerate(zip(MODELS, colors)):
    pf = baseline[m]['mse_per_freq']
    means = [pf[str(k)]['mean'] for k in range(4)]
    stds  = [pf[str(k)]['std']  for k in range(4)]
    ax.bar(x + i*width, means, width, yerr=stds, label=m.upper(),
           color=col, capsize=4, alpha=0.85, edgecolor='black', linewidth=0.7)
ax.set_xticks(x + width)
ax.set_xticklabels(FREQ_LABELS)
ax.set_ylabel('MSE (mean ± std, 3 seeds)')
ax.set_title('Per-Frequency MSE by Architecture (Baseline)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.show()

### 5.1 Discussion

The per-frequency MSE pattern reveals an important physical insight:

- **High-frequency components (300 Hz) are often *easier* to extract**, not
  harder. At 300 Hz, three full cycles fit in the 10-sample window — the
  signal presents a highly distinctive, rapidly oscillating pattern that all
  models can identify reliably.

- **Low-frequency components (10 Hz) are harder** because only 0.1 cycles are
  visible per window — essentially a slow linear trend indistinguishable from
  amplitude noise without the selector context.

This **inverts the vanishing-gradient hypothesis** for this specific task:
the information content per window increases with frequency, giving high-frequency
extraction a natural advantage independent of architecture.

---
## 6. Noise Robustness (EXP-02)

**Protocol:** Amplitude noise $\alpha \in \{0.00, 0.05, 0.10, 0.20, 0.40, 0.80\}$,
$\beta$ fixed at 0.1, seed averaged over 3 runs.

**Breaking point** is defined as the smallest $\alpha$ where
$\text{MSE} > 10 \times \text{MSE}_{\text{baseline}}$ for any model.

In [ ]:
alpha_dir = os.path.join(RESULTS, 'noise_sweep', 'alpha')
canonical = ['alpha_0_00', 'alpha_0_05', 'alpha_0_10',
             'alpha_0_20', 'alpha_0_40', 'alpha_0_80']
canonical = [c for c in canonical
             if os.path.exists(os.path.join(alpha_dir, c, 'summary.json'))]
alpha_vals_map = {'alpha_0_00': 0.00, 'alpha_0_05': 0.05, 'alpha_0_10': 0.10,
                  'alpha_0_20': 0.20, 'alpha_0_40': 0.40, 'alpha_0_80': 0.80}
alpha_x = [alpha_vals_map[c] for c in canonical]

noise_data = {m: {'means': [], 'stds': []} for m in MODELS}
for cond in canonical:
    s = load_json(os.path.join(alpha_dir, cond, 'summary.json'))['models']
    for m in MODELS:
        noise_data[m]['means'].append(s[m]['mse_overall']['mean'])
        noise_data[m]['stds'].append(s[m]['mse_overall']['std'])

fig, ax = plt.subplots(figsize=(8, 5))
for m, col in zip(MODELS, ['#4C72B0', '#DD8452', '#55A868']):
    ax.errorbar(alpha_x, noise_data[m]['means'], yerr=noise_data[m]['stds'],
                marker='o', label=m.upper(), color=col, capsize=4, linewidth=1.8)
ax.set_xlabel(r'Amplitude noise coefficient $\alpha$')
ax.set_ylabel('Test MSE (mean ± std)')
ax.set_title(r'Noise Robustness — MSE vs $\alpha$ ($\beta$=0.1)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'noise_heatmap.png')))

### 6.1 Discussion

**Key observations:**

- At low noise ($\alpha \leq 0.10$) all three architectures achieve similarly low
  MSE — the task is easy enough that architecture choice is secondary.

- At high noise ($\alpha = 0.80$) **RNN degrades most sharply**, consistent with
  the absence of a gating mechanism to suppress noisy signals.

- **LSTM** shows the most consistent performance across noise levels, owing to
  the forget gate's ability to attenuate unreliable activations:

  $$\mathbf{f}_t = \sigma\!\bigl(W_f[\mathbf{h}_{t-1}, \mathbf{x}_t] + b_f\bigr)$$

  When $\mathbf{f}_t \approx 0$, the previous cell state is discarded;
  when $\mathbf{f}_t \approx 1$, it is preserved. This selective memory
  protects the cell state $\mathbf{c}_t$ from noise-corrupted inputs.

- **FC** performance at high noise is competitive with LSTM, because
  it has no recurrent dependency to amplify noise over sequential steps.

---
## 7. Hidden-Size Sensitivity (EXP-04)

**Research question (RQ-4):** Does increasing model capacity monotonically
improve MSE, or is there a point of diminishing returns?

**Protocol:** `hidden_size` $\in \{64, 128, 256, 512\}$, `n_layers=2`, seed=42.

In [ ]:
hidden_sizes = [64, 128, 256, 512]
hs_data = {m: {'means': [], 'stds': []} for m in MODELS}
for h in hidden_sizes:
    s = load_json(os.path.join(RESULTS, 'hidden_size', f'h{h}', 'summary.json'))['models']
    for m in MODELS:
        hs_data[m]['means'].append(s[m]['mse_overall']['mean'])
        hs_data[m]['stds'].append(s[m]['mse_overall']['std'])

# Approximate parameter counts
def fc_params(H):   return 14*H+H + H*H+H + H*10+10
def rnn_params(H):  return 14*H+H + 2*(H*H+H*H+H) + H*10+10
def lstm_params(H): return 14*H+H + 4*2*(H*H+H) + H*10+10
pfns = {'fc': fc_params, 'rnn': rnn_params, 'lstm': lstm_params}

print(f"{'Arch':<6}  " + '  '.join(f'H={h}  params       MSE' for h in hidden_sizes))
print('-'*100)
for m in MODELS:
    row = '  '.join(f'{pfns[m](h):>8,}  {hs_data[m]["means"][i]:.2e}'
                    for i, h in enumerate(hidden_sizes))
    print(f"{m.upper():<6}  {row}")

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'sensitivity_hidden_size.png')))

### 7.1 Discussion

Parameter counts grow as:

| Architecture | Dominant term | Relative size (H=128) |
|---|---|---|
| FC   | $H^2$ (hidden-to-hidden layer) | ~18K |
| RNN  | $2H^2$ per layer (input→hidden, hidden→hidden) | ~48K |
| LSTM | $8H^2$ per layer (four gates × two matrices) | ~135K |

**LSTM has ~4–7× more parameters** than FC/RNN at the same `hidden_size`,
which means fair comparisons require parameter-matched configurations
(EXP-03, not yet run).

**Key finding:** No monotonic relationship between hidden size and MSE.
The task complexity is low (14-dimensional input, stationary signals) — the
capacity sweet spot is H=64–128. Larger models risk over-parameterisation
and unstable training at the batch size used (256).

---
## 8. Depth Sensitivity (EXP-05)

**Protocol:** `n_layers` $\in \{1, 2, 3, 4\}$, `hidden_size=128`, seed=42.

The gradient flowing back through $L$ stacked RNN layers satisfies:

$$\left\|\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0^{(1)}}\right\| \leq
\prod_{l=1}^{L} \left\|W_{hh}^{(l)\top}\right\| \cdot \left\|\text{diag}(\sigma'(\cdot))\right\|$$

If $\|W_{hh}\| < 1$ this product shrinks exponentially in $L$, degrading training.

In [ ]:
n_layers_list = [1, 2, 3, 4]
nl_data = {m: [] for m in MODELS}
for l in n_layers_list:
    s = load_json(os.path.join(RESULTS, 'n_layers', f'L{l}', 'summary.json'))['models']
    for m in MODELS:
        nl_data[m].append(s[m]['mse_overall']['mean'])

print(f"{'Model':<6}  " + '  '.join(f'L={l}' for l in n_layers_list))
print('-' * 40)
for m in MODELS:
    vals = '  '.join(f'{v:.2e}' for v in nl_data[m])
    print(f"{m.upper():<6}  {vals}")

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'sensitivity_n_layers.png')))

### 8.1 Discussion

- **FC**: Relatively insensitive to depth. The universal approximation
  theorem guarantees that a single hidden layer of sufficient width can
  approximate any continuous function — additional layers add capacity
  but not necessarily accuracy on this task.

- **RNN**: Erratic performance — stacking RNN layers compounds the
  vanishing-gradient problem rather than resolving it.

- **LSTM**: L=1 often matches or beats deeper configurations. Deep LSTMs
  risk over-parameterisation on this low-complexity task.

**Recommendation:** Use L=1 or L=2 for all architectures on this task.

---
## 9. Learning-Rate Sensitivity (EXP-06)

**Protocol:** `lr` $\in \{10^{-4}, 10^{-3}, 3\times10^{-3}, 10^{-2}\}$,
`hidden_size=128`, `n_layers=2`, seed=42.

The Adam optimiser update:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}, \qquad
\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

In [ ]:
lr_dirs = ['lr_0_0001', 'lr_0_0010', 'lr_0_0030', 'lr_0_0100']
lr_vals = [1e-4, 1e-3, 3e-3, 1e-2]
lr_data = {m: [] for m in MODELS}
for d in lr_dirs:
    s = load_json(os.path.join(RESULTS, 'lr_sweep', d, 'summary.json'))['models']
    for m in MODELS:
        lr_data[m].append(s[m]['mse_overall']['mean'])

print(f"{'Model':<6}  " + '  '.join(f'lr={v:.0e}' for v in lr_vals))
print('-' * 55)
for m in MODELS:
    vals = '  '.join(f'{v:.2e}' for v in lr_data[m])
    print(f"{m.upper():<6}  {vals}")

In [ ]:
display(Image(filename=os.path.join(ASSETS, 'sensitivity_lr.png')))

### 9.1 Discussion

- `lr=1e-4`: Under-convergence within 100 epochs. MSE is elevated for all models.
- `lr=1e-3` (default): Good balance for all architectures — lowest or near-lowest
  MSE in most cases.
- `lr=3e-3`: Acceptable for FC; LSTM begins to show instability.
- `lr=1e-2`: Clear divergence/oscillation for FC and RNN (large MSE); LSTM
  is more tolerant, possibly because its gating provides implicit gradient
  stabilisation.

**Recommendation:** `lr=1e-3` is the robust default for Adam on this task
across all three architectures.

---
## 10. Conclusions

### 10.1 Research Question Answers

| RQ | Question | Finding |
|---|---|---|
| RQ-1 | Which architecture achieves lowest MSE? | **FC** at default settings; **LSTM** competitive at optimal H |
| RQ-2 | At what noise level does extraction break? | RNN degrades first ($\alpha=0.40$–$0.80$); LSTM most robust |
| RQ-3 | Architecture vs. capacity gap? | LSTM has 4–7× more params at same H — parameter-matched comparison (EXP-03) needed |
| RQ-4 | Sensitivity to hidden size / depth / LR? | H=128, L=1–2, lr=1e-3 is the sweet spot for all architectures |
| RQ-5 | RNN vanishing gradient at 300 Hz? | **Hypothesis refuted for this task** — 300 Hz is *easier* due to more cycles per window |

### 10.2 Architecture Comparison Summary

| Criterion | FC | RNN | LSTM |
|---|---|---|---|
| Baseline MSE | **Lowest** | Highest | Middle |
| Noise robustness ($\alpha=0.80$) | Good | **Worst** | **Best** |
| Depth sensitivity | Low | High | Medium |
| Training stability | **High** | Low | High |
| Parameters (H=128, L=2) | ~18K | ~48K | ~135K |
| When to prefer | Short window, clean/low-noise | Not recommended | High noise, long-range dependencies |

### 10.3 Theoretical Interpretation

The dominant factor in this experiment is the **input encoding**: concatenating the
full window as a flat vector (seq_len=1) removes the temporal advantage of
recurrent models. If the input were processed step-by-step (seq_len=10), the
ranking would likely favour LSTM — a follow-up experiment worth pursuing.

The noise robustness advantage of LSTM is attributable to the forget gate:

$$\mathbf{f}_t = \sigma(W_f[\mathbf{h}_{t-1}, \mathbf{x}_t] + b_f) \in (0,1)^H$$

Small $f_t$ values suppress unreliable features, providing a learned noise filter
that FC and RNN lack.

### 10.4 Practical Recommendation

| Scenario | Recommended architecture |
|---|---|
| Low noise, short window | **FC** (H=128, simplest and fastest) |
| High noise ($\alpha > 0.4$) | **LSTM** (H=64–128, best noise robustness) |
| Real-world deployment | **LSTM** (safe default; graceful degradation under noise) |

In [ ]:
# Final overall-MSE bar chart with error bars
means = [baseline[m]['mse_overall']['mean'] for m in MODELS]
stds  = [baseline[m]['mse_overall']['std']  for m in MODELS]
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([m.upper() for m in MODELS], means, yerr=stds,
              color=colors, alpha=0.85, edgecolor='black',
              linewidth=0.8, capsize=6)
ax.set_ylabel('Overall Test MSE (mean ± std, 3 seeds)')
ax.set_title('Final Baseline Comparison — FC vs RNN vs LSTM')
ax.set_yscale('log')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean * 1.5,
            f'{mean:.1e}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print('Conclusion:')
best = MODELS[np.argmin(means)]
print(f'  Best architecture (default config): {best.upper()}')
print(f'  Most noise-robust: LSTM')
print(f'  Least recommended: RNN')